## Dataset Structure

The division of data within the Crack Segmentation Dataset is outlined as follows:

- **Training set**: Consists of 3717 images with corresponding annotations.

- **Testing set**: Comprises 112 images along with their respective annotations.

- **Validation set**: Includes 200 images with their corresponding annotations.

In [1]:
import ultralytics
ultralytics.checks()

Ultralytics 8.4.47  Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Setup complete  (32 CPUs, 31.7 GB RAM, 706.2/1661.9 GB disk)


In [4]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26m-seg.pt")  # load a pretrained model (recommended for training)

WARNING Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26m-seg.pt... <urlopen error [SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1010)>
WARNING Download failure, retrying 2/3 https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26m-seg.pt... Curl return value 35


In [3]:
# Train the model
results = model.train(
    data=r"D:\Develop\dataset\crack-multi-seg\data.yaml",
    epochs=3,
    imgsz=640,
    batch=64,
    workers=0,
)

New https://pypi.org/project/ultralytics/8.4.54 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.47  Python-3.12.13 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\Develop\dataset\crack-multi-seg\data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0

KeyboardInterrupt: 

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

# Load a model
modelp = YOLO("runs/segment/train-2/weights/best.pt")  # load a fine-tuned model

# Inference using the model (img/video/stream)
prediction_results = modelp.predict(
    r"D:\HP\Downloads\12687.jpg",
    save=False,
    conf=0.01,
)

plt.imshow(prediction_results[0].plot()[..., ::-1])  # 转换 BGR 到 RGB
plt.show()

In [ ]:
# 在 notebook 顶部加一行确保 inline 显示
%matplotlib inline

from matplotlib import pyplot as plt
import numpy as np

# 诊断
print("prediction_results 类型:", type(prediction_results), "长度:", len(prediction_results))
if len(prediction_results)==0:
    raise RuntimeError("prediction_results 为空，先确认 model.predict() 的输入路径和 conf 阈值")

res = prediction_results[0]
print("has boxes:", hasattr(res, "boxes"), "mask count:", getattr(res, "masks", None) is not None)

# 获取带标注的图像并显示（res.plot() 返回图像数组）
img = res.plot()  # numpy array (RGB)
plt.figure(figsize=(10,8))
plt.imshow(img)
plt.axis("off")
plt.show()

# 提取几何特征

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage.morphology import skeletonize
import pandas as pd

In [ ]:
# 获取第一个预测结果
result = prediction_results[0]

# 获取原图（numpy 数组，BGR 格式）
img = result.orig_img

# 获取所有 bbox（xyxy 格式：x1, y1, x2, y2）
boxes = result.boxes.xyxy.cpu().numpy().astype(int)

# 截取第一个 bbox 对应的图像区域
x1, y1, x2, y2 = boxes[0]
bbox_img = img[y1:y2, x1:x2]

print(f"bbox 坐标: ({x1}, {y1}, {x2}, {y2})")
print(f"截取图像尺寸: {bbox_img.shape}")
plt.imshow(bbox_img[:, :, ::-1])  # 转换 BGR 到 RGB

cv2.imwrite("bbox_img.png", bbox_img)

In [ ]:
# ========== 获取 mask ==========
masks = result.masks.data.cpu().numpy()  # (N, H, W) 二值 mask
mask = masks[0]  # 取第一个检测目标的 mask

# 1. mask 面积
mask_area = int(mask.sum())

# 2. 面积比（mask面积 / bbox面积）
bbox_w, bbox_h = x2 - x1, y2 - y1
bbox_area = bbox_w * bbox_h
rectangularity = mask_area / bbox_area if bbox_area > 0 else 0

# 3. 细长度（最小外接矩形的长宽比）
mask_uint8 = (mask * 255).astype(np.uint8)
contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
if contours:
    largest_contour = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(largest_contour)
    rw, rh = rect[1]
    slenderness = max(rw, rh) / min(rw, rh) if min(rw, rh) > 0 else 0
else:
    slenderness = 0
    rect = None

# 4. 主方向（最小外接矩形的角度）
if rect is not None:
    major_direction = rect[2]
    if rect[1][0] < rect[1][1]:
        major_direction = 90 + major_direction
else:
    major_direction = 0

# 5. 边界复杂度（周长² / 面积）
if contours:
    perimeter = cv2.arcLength(largest_contour, True)
    boundary_complexity = (perimeter ** 2) / mask_area if mask_area > 0 else 0
else:
    boundary_complexity = 0

# 6. 骨架长度 & 7. 分支数量
skel = skeletonize(mask > 0).astype(np.uint8)
skeleton_length = int(skel.sum())

branch_points = 0
end_points = 0
h, w = skel.shape
for i in range(1, h - 1):
    for j in range(1, w - 1):
        if skel[i, j] == 1:
            neighbors = skel[i-1:i+2, j-1:j+2].sum() - 1
            if neighbors >= 3:
                branch_points += 1
            elif neighbors == 1:
                end_points += 1

# 8. 区域颜色统计
# mask 尺寸可能与原图不一致，需缩放到原图大小
if mask.shape != img.shape[:2]:
    mask_full = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
else:
    mask_full = mask
mask_bool = mask_full.astype(bool)
pixel_values = img[mask_bool]  # (N, 3) BGR
color_mean = pixel_values.mean(axis=0)
color_std = pixel_values.std(axis=0)

# ========== 输出结果 ==========
features = {
    "mask面积": mask_area,
    "面积比": round(rectangularity, 4),
    "细长度": round(slenderness, 4),
    "主方向(°)": round(major_direction, 2),
    "边界复杂度": round(boundary_complexity, 4),
    "骨架长度": skeleton_length,
    "分支数量": branch_points,
    "BGR均值": [round(v, 2) for v in color_mean],
    "BGR标准差": [round(v, 2) for v in color_std],
}

df = pd.DataFrame([features])
df